In [23]:
import pandas as pd
import numpy as np
from datetime import datetime

# ── 1. LOAD DATA ─────────────────────────────────────────────────────────────
cancellation = pd.read_csv("cancellation.csv")
subscription = pd.read_csv("subscription_v2.csv", sep=";")

# ── 2. PRE-MERGE CLEANING ────────────────────────────────────────────────────
# Drop duplicates in cancellation first to avoid ballooning the merge
cancellation = cancellation.drop_duplicates(subset="pseudo_id", keep="first")

# ── 3. MERGE ─────────────────────────────────────────────────────────────────
# Merge early so we have access to all columns for logic
df = pd.merge(subscription, cancellation, on="pseudo_id", how="left")

# ── 4. DATE PARSING ──────────────────────────────────────────────────────────
# DMY columns (standardizing all date formats)
date_cols_dmy = ["subscription_cancel_date", "first_campaign_day", "last_campaign_day", "usr_created", "birthdate"]
for col in date_cols_dmy:
    df[col] = pd.to_datetime(df[col], dayfirst=True, errors="coerce")

# Remaining date columns
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
df["expiration_date"] = pd.to_datetime(df["expiration_date"], errors="coerce")

# ── 5. KEEP "ACTIVE" RECORD (SORT & DROP) ────────────────────────────────────
# Sort by expiration_date descending (furthest in future at top)
df = df.sort_values(by="expiration_date", ascending=False)
# Now drop duplicates so we keep the most "active" or "latest" entry per user
df = df.drop_duplicates(subset="pseudo_id", keep="first")

# ── 6. CHURN LOGIC ───────────────────────────────────────────────────────────
# Churn: did NOT continue (expiration is on or before the last campaign day)
df['churn'] = 0
mask_churn = (df['expiration_date'].notna()) & (df['expiration_date'] <= df['last_campaign_day'])
df.loc[mask_churn, 'churn'] = 1

# Continued subscription (inverse of churn)
df['continued_subscription'] = 1 - df['churn']

# Early Churn: Continued, but expired within 90 days of campaign end
df['early_churn'] = 0
ninety_days_post = df['last_campaign_day'] + pd.Timedelta(days=90)
mask_early = (df['continued_subscription'] == 1) & \
             (df['expiration_date'].notna()) & \
             (df['expiration_date'] <= ninety_days_post)
df.loc[mask_early, 'early_churn'] = 1

# ── 7. CLEANING & FEATURE ENGINEERING ────────────────────────────────────────
# Drop rows missing critical dates for age/loyalty calculations
df = df.dropna(subset=['birthdate', 'usr_created', 'order_date'])

# Fill NA for categorical columns
df['type'] = df['type'].fillna("Ingen afmelding")
df['reason'] = df['reason'].fillna("Ingen afmelding")

# Replace missing expiration with "Future" date (Using 2100 to avoid Overflow)
df['expiration_date'] = df['expiration_date'].fillna(pd.Timestamp("2100-01-01"))

# Permissions (Convert "true" string to 1/0)
df['permission_given_order'] = df['permission_given_order'].apply(lambda x: 1 if str(x).lower() == "true" else 0)
df['permission_given_today'] = df['permission_given_today'].apply(lambda x: 1 if str(x).lower() == "true" else 0)

# Derived variables: Age and Kundetid
today = pd.Timestamp(datetime.now())
df['age'] = (today - df['birthdate']).dt.days / 365.25
df['kundetid_dage'] = (df['order_date'] - df['usr_created']).dt.days

# Filter invalid kundetid (ensure they were created before they ordered)
df = df[df['kundetid_dage'] >= 0]

# Kundetid groups (Equivalent to case_when)
bins = [-np.inf, 7, 30, 180, np.inf]
labels = ["0-7 dage", "8-30 dage", "1-6 måneder", "6+ måneder"]
df['kundetid_gruppe'] = pd.cut(df['kundetid_dage'], bins=bins, labels=labels)

# ── 8. FINAL SELECTION & CASTING ─────────────────────────────────────────────
# Drop raw columns used for features
cols_to_drop = ['kundetid_dage', 'order_date', 'birthdate', 'usr_created']
model_data = df.drop(columns=cols_to_drop).copy()

# Cast binary and factor columns to 'category'
cat_cols = [
    "churn", "type", "reason", "koen", "kundetid_gruppe",
    "permission_given_order", "permission_given_today",
    "continued_subscription", "early_churn"
]

for col in cat_cols:
    if col in model_data.columns:
        model_data[col] = model_data[col].astype("category")

# ── 9. OUTPUT ────────────────────────────────────────────────────────────────
print("--- Churn Distribution ---")
print(model_data['churn'].value_counts(normalize=True))

print("\n--- Summary ---")
print(f"Total rows in model_data: {len(model_data)}")
print(model_data.info())


--- Churn Distribution ---
churn
0    0.58011
1    0.41989
Name: proportion, dtype: float64

--- Summary ---
Total rows in model_data: 1267
<class 'pandas.core.frame.DataFrame'>
Index: 1267 entries, 286 to 1276
Data columns (total 22 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   pseudo_id                 1267 non-null   object        
 1   account_active_days       1267 non-null   int64         
 2   subscription_cancel_date  1049 non-null   datetime64[ns]
 3   koen                      1240 non-null   category      
 4   order_trackertag          1261 non-null   object        
 5   permission_given_order    1267 non-null   category      
 6   permission_given_today    1267 non-null   category      
 7   previous_subscriptions    1267 non-null   int64         
 8   previous_campaigns        1267 non-null   int64         
 9   previous_trials           1267 non-null   int64         
 10  first_c

In [26]:
# model_01

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, backend as K

# 1. Define specific columns based on our model_data
target_col = "churn"

# These are already 0/1 categories, so we just treat them as numbers
dummy_cols = ["permission_given_order", "permission_given_today"]

# These need One-Hot Encoding
cat_cols = ["koen", "type", "reason", "kundetid_gruppe"]

# These need scaling
quant_cols = [
    "age", "previous_subscriptions", "previous_campaigns",
    "previous_trials", "newsletters_before_order", "newsletters_after_order"
]

# 2. Build Feature Matrix
y = model_data[target_col].astype(int).values

# One-hot encoding the multi-level categories
df_dummies = pd.get_dummies(model_data[cat_cols], drop_first=False)

# Combine: Dummies | One-Hot | Quantitative
X_df = pd.concat([
    model_data[dummy_cols].astype(float),
    df_dummies.astype(float),
    model_data[quant_cols]
], axis=1)

# 3. Split into Train/Test (80/20)
X_train_df, X_test_df, y_train, y_test = train_test_split(
    X_df, y, test_size=0.20, random_state=42, stratify=y
)

# 4. Standardize ONLY quantitative columns
scaler = StandardScaler()
X_train_arr = X_train_df.copy()
X_test_arr = X_test_df.copy()

X_train_arr[quant_cols] = scaler.fit_transform(X_train_df[quant_cols])
X_test_arr[quant_cols] = scaler.transform(X_test_df[quant_cols])

# Convert to numpy float32 for Keras
X_train = X_train_arr.values.astype(np.float32)
X_test = X_test_arr.values.astype(np.float32)

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import regularizers

# 5. Build a Robust Keras Model
K.clear_session()
tf.random.set_seed(42)

model_02 = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),

    # Layer 1 + Dropout
    layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.3),

    # Layer 2 + Dropout
    layers.Dense(32, activation="relu", kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.2),

    layers.Dense(1, activation="sigmoid")
], name="robust_churn_model")

model_02.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
             tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")]
)

# 6. Define Early Stopping
# This stops training if val_loss doesn't improve for 10 epochs
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# 7. Train the model
history = model_02.fit(
    X_train, y_train,
    epochs=150, # High number, but early_stop will handle it
    batch_size=32, # Smaller batch size often generalizes better
    validation_split=0.20,
    callbacks=[early_stop],
    verbose=1
)

# 8. Evaluate
metrics = model_02.evaluate(X_test, y_test, verbose=0)
print("\n" + "="*30)
print(f"Test AUC: {metrics[2]:.4f} (Focus on this!)")
print(f"Test Accuracy:  {metrics[1]:.4f}")
print(f"Test Recall:    {metrics[4]:.4f}")
print("="*30)


Epoch 1/150
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.4901 - auc: 0.5332 - loss: 0.7947 - precision: 0.4319 - recall: 0.6472 - val_accuracy: 0.6305 - val_auc: 0.6696 - val_loss: 0.7472 - val_precision: 0.6296 - val_recall: 0.2073
Epoch 2/150
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5877 - auc: 0.6042 - loss: 0.7519 - precision: 0.5222 - recall: 0.3090 - val_accuracy: 0.6798 - val_auc: 0.7586 - val_loss: 0.7098 - val_precision: 0.8148 - val_recall: 0.2683
Epoch 3/150
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6173 - auc: 0.6670 - loss: 0.7243 - precision: 0.5740 - recall: 0.3732 - val_accuracy: 0.7094 - val_auc: 0.7918 - val_loss: 0.6728 - val_precision: 0.7556 - val_recall: 0.4146
Epoch 4/150
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6481 - auc: 0.7277 - loss: 0.6870 - precision: 0.6133 - recall: 0.4577 - val_accuracy: 0.7094 - val_auc: 0.8040 - val_loss: 0.6390 - val_precision: 0.7018 - val_recall: 0.4878
Epoch 5/150
26/26 ━━━━━━━━━━━━━━━━━

In [ ]:
# model_02

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, backend as K
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import regularizers

# 1. Define specific columns
target_col = "churn"
dummy_cols = ["permission_given_order", "permission_given_today"]
cat_cols = ["koen", "type", "reason", "kundetid_gruppe"]
quant_cols = ["age", "previous_subscriptions", "previous_campaigns",
              "previous_trials", "newsletters_before_order", "newsletters_after_order"]

# 2. Build Feature Matrix
y = model_data[target_col].astype(int).values
df_dummies = pd.get_dummies(model_data[cat_cols], drop_first=False)
X_df = pd.concat([
    model_data[dummy_cols].astype(float),
    df_dummies.astype(float),
    model_data[quant_cols]
], axis=1)

# 3. Split into Train/Test
X_train_df, X_test_df, y_train, y_test = train_test_split(
    X_df, y, test_size=0.20, random_state=42, stratify=y
)

# 4. Standardize
scaler = StandardScaler()
X_train_arr = X_train_df.copy()
X_test_arr = X_test_df.copy()
X_train_arr[quant_cols] = scaler.fit_transform(X_train_df[quant_cols])
X_test_arr[quant_cols] = scaler.transform(X_test_df[quant_cols])

X_train = X_train_arr.values.astype(np.float32)
X_test = X_test_arr.values.astype(np.float32)

# 5. Build the "Stronger" Model
K.clear_session()
tf.random.set_seed(42)

model_final = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),

    # Layer 1: Wider initial layer to capture interactions
    layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(0.002)),
    layers.BatchNormalization(), # Added for stability
    layers.Dropout(0.4),

    # Layer 2: Medium layer
    layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(0.002)),
    layers.Dropout(0.3),

    # Layer 3: Final refinement layer
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.2),

    layers.Dense(1, activation="sigmoid")
], name="stronger_churn_model")

# Using a slightly lower starting learning rate as seen in advanced notebook examples
optimizer = keras.optimizers.Adam(learning_rate=0.001)

model_final.compile(
    optimizer=optimizer,
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
             tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")]
)

# 6. Advanced Callbacks
# EarlyStopping: Restores weights from the absolute best epoch
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

# ReduceLROnPlateau: If the model stops improving, make smaller "steps"
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.00001)

# 7. Train the model
history = model_final.fit(
    X_train, y_train,
    epochs=200,
    batch_size=32,
    validation_split=0.20,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# 8. Final Evaluation
metrics = model_final.evaluate(X_test, y_test, verbose=0)
print("\n" + "="*40)
print(f"Test AUC:       {metrics[2]:.4f}")
print(f"Test Accuracy:  {metrics[1]:.4f}")
print(f"Test Recall:    {metrics[4]:.4f}")
print("="*40)

Epoch 1/200
26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - accuracy: 0.5210 - auc: 0.5140 - loss: 1.1006 - precision: 0.4272 - recall: 0.3848 - val_accuracy: 0.6650 - val_auc: 0.7331 - val_loss: 0.9503 - val_precision: 0.6346 - val_recall: 0.4024 - learning_rate: 0.0010
Epoch 2/200
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5877 - auc: 0.6274 - loss: 0.9536 - precision: 0.5135 - recall: 0.4985 - val_accuracy: 0.6995 - val_auc: 0.7747 - val_loss: 0.9254 - val_precision: 0.6780 - val_recall: 0.4878 - learning_rate: 0.0010
Epoch 3/200
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6506 - auc: 0.7080 - loss: 0.8805 - precision: 0.5909 - recall: 0.5685 - val_accuracy: 0.7044 - val_auc: 0.7881 - val_loss: 0.8988 - val_precision: 0.6964 - val_recall: 0.4756 - learning_rate: 0.0010
Epoch 4/200
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6494 - auc: 0.7245 - loss: 0.8460 - precision: 0.5931 - recall: 0.5481 - val_accuracy: 0.6897 - val_auc: 0.7933 - val_loss: 0.8729 - val_pr